In [1]:
from pathlib import Path
import gc
import importlib
import json
import os
import re
import sys
import time

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
HF_COMPAT_DIR = PROJECT_ROOT / '.hf_compat'
if HF_COMPAT_DIR.exists():
    compat_path = str(HF_COMPAT_DIR)
    if compat_path in sys.path:
        sys.path.remove(compat_path)
    sys.path.insert(0, compat_path)
    importlib.invalidate_caches()

import numpy as np
import pandas as pd
import sacrebleu
import torch
from peft import PeftModel
from tqdm.auto import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

LOCAL_BASE_MODEL = Path(r'C:\Users\WingYouther\.cache\huggingface\hub\models--facebook--m2m100_418M\snapshots\55c2e61bbf05dfb8d7abccdc3fae6fc8512fd636')
BASE_MODEL = str(LOCAL_BASE_MODEL) if (LOCAL_BASE_MODEL / 'config.json').exists() else 'facebook/m2m100_418M'
ADAPTER_PATH = PROJECT_ROOT / 'models' / 'lora' / 'm2m100_en_uz_public_v2' / 'final_adapter'
BENCHMARK_FILE = PROJECT_ROOT / 'results' / 'trusted_benchmark_eval' / 'benchmark_pairs.csv'
OLD_METRICS_FILE = PROJECT_ROOT / 'results' / 'trusted_benchmark_eval' / 'metrics_all.csv'
RESULT_DIR = PROJECT_ROOT / 'results' / 'trusted_benchmark_public_v2'
RESULT_DIR.mkdir(parents=True, exist_ok=True)
FALLBACK_CACHE_DIR = PROJECT_ROOT / 'models' / 'huggingface' / 'hub'
MAX_SOURCE_LENGTH = 128
MAX_NEW_TOKENS = 128
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.bfloat16 if DEVICE == 'cuda' and torch.cuda.is_bf16_supported() else (torch.float16 if DEVICE == 'cuda' else torch.float32)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
assert ADAPTER_PATH.exists(), f'Adapter not found: {ADAPTER_PATH}'
assert BENCHMARK_FILE.exists(), 'Run Notebook 10 through benchmark construction first.'
assert OLD_METRICS_FILE.exists(), 'Old base/adapter metrics are required for comparison.'
assert DEVICE == 'cuda', 'CUDA GPU was not detected. Select the project virtual-environment kernel.'
print('Device:', DEVICE, 'dtype:', DTYPE)
print('Base model:', BASE_MODEL)
print('New adapter:', ADAPTER_PATH)
print('Output:', RESULT_DIR)


D:\dev\projects\fourlang_translation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda dtype: torch.bfloat16
Base model: C:\Users\WingYouther\.cache\huggingface\hub\models--facebook--m2m100_418M\snapshots\55c2e61bbf05dfb8d7abccdc3fae6fc8512fd636
New adapter: D:\dev\projects\fourlang_translation\models\lora\m2m100_en_uz_public_v2\final_adapter
Output: D:\dev\projects\fourlang_translation\results\trusted_benchmark_public_v2


In [2]:
benchmark_df = pd.read_csv(BENCHMARK_FILE, keep_default_na=False)
required = {'eval_id', 'benchmark', 'pair_id', 'src_lang', 'tgt_lang', 'source', 'reference'}
missing = required.difference(benchmark_df.columns)
assert not missing, f'Missing benchmark columns: {sorted(missing)}'
assert len(benchmark_df) == 1200, f'Expected the fixed 1200-row benchmark, got {len(benchmark_df)}'
assert not benchmark_df['eval_id'].duplicated().any(), 'Duplicate eval_id found.'
assert not benchmark_df[['source', 'reference']].apply(lambda s: s.astype(str).str.strip().eq('')).any().any(), 'Empty text found.'
counts = benchmark_df.groupby(['benchmark', 'src_lang', 'tgt_lang']).size().rename('samples').reset_index()
assert counts['samples'].eq(200).all(), counts
display(counts)
display(benchmark_df.head())


,benchmark,src_lang,tgt_lang,samples
0,flores_devtest,en,uz,200
1,flores_devtest,uz,en,200
2,ntrex,en,uz,200
3,ntrex,uz,en,200
4,tatoeba_latin,en,uz,200
5,tatoeba_latin,uz,en,200


,eval_id,benchmark,pair_id,src_lang,tgt_lang,source,reference
0,eval_000000,flores_devtest,1008,en,uz,"As the areas are sparsely populated, and light...","Hududlarda aholi siyrakligi tufayli, yorug'lik..."
1,eval_000001,flores_devtest,1011,en,uz,"Workplace harmony is crucial, emphasizing grou...",Alohida shaxslarning yutuqlarini maqtashdan ko...
2,eval_000002,flores_devtest,102,en,uz,"However, the percentage of XDR-TB in the entir...",Ammo sil kasalligi bilan xastalangan odamlarni...
3,eval_000003,flores_devtest,108,en,uz,A doctor who worked at Children's Hospital of ...,Pensilvaniya shtati Pittsburg shahridagi bolal...
4,eval_000004,flores_devtest,11,en,uz,While one experimental vaccine appears able to...,Birgina tajriba vaksinasi Eboladan o'lish xavf...


In [3]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, local_files_only=Path(BASE_MODEL).exists(), cache_dir=str(FALLBACK_CACHE_DIR))

def load_adapter_model():
    kwargs = {'torch_dtype': DTYPE, 'low_cpu_mem_usage': True}
    if BASE_MODEL == 'facebook/m2m100_418M':
        kwargs['cache_dir'] = str(FALLBACK_CACHE_DIR)
    else:
        kwargs['local_files_only'] = True
    base = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL, **kwargs)
    model = PeftModel.from_pretrained(base, ADAPTER_PATH)
    model.to(DEVICE)
    model.eval()
    model.config.use_cache = True
    return model

def synchronize():
    if DEVICE == 'cuda':
        torch.cuda.synchronize()

def word_tokens(text):
    return re.findall(r"[^\W_]+(?:['’ʻʼ-][^\W_]+)*", str(text).casefold(), flags=re.UNICODE)

def repeated_phrase(text):
    tokens = word_tokens(text)
    for width in range(1, min(6, len(tokens) // 3 + 1)):
        for start in range(len(tokens) - 3 * width + 1):
            phrase = tokens[start:start + width]
            if phrase == tokens[start + width:start + 2 * width] == tokens[start + 2 * width:start + 3 * width]:
                return True
    return False

@torch.inference_mode()
def translate_one(model, text, src_lang, tgt_lang):
    tokenizer.src_lang = src_lang
    started_total = time.perf_counter()
    inputs = tokenizer(str(text), return_tensors='pt', truncation=True, max_length=MAX_SOURCE_LENGTH).to(DEVICE)
    synchronize()
    generated = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.get_lang_id(tgt_lang),
        max_new_tokens=MAX_NEW_TOKENS,
        num_beams=1,
        repetition_penalty=1.15,
        no_repeat_ngram_size=3,
    )
    synchronize()
    total_seconds = time.perf_counter() - started_total
    prediction = tokenizer.batch_decode(generated, skip_special_tokens=True)[0]
    return prediction, total_seconds, int(generated.shape[-1])

print('Evaluation helpers are ready.')


Evaluation helpers are ready.


In [4]:
PREDICTION_FILE = RESULT_DIR / 'predictions_public_v2_adapter.csv'
if PREDICTION_FILE.exists():
    existing = pd.read_csv(PREDICTION_FILE, keep_default_na=False)
else:
    existing = pd.DataFrame()
completed = set(existing['eval_id'].astype(str)) if len(existing) else set()
pending = benchmark_df[~benchmark_df['eval_id'].astype(str).isin(completed)]
print('Completed:', len(completed), 'Pending:', len(pending))

if len(pending):
    if 'model' in globals():
        del model
    gc.collect()
    torch.cuda.empty_cache()
    model = load_adapter_model()
    _ = translate_one(model, 'Hello.', 'en', 'uz')
    new_rows = []
    for row in tqdm(pending.itertuples(index=False), total=len(pending), desc='public_v2_adapter'):
        prediction, total_seconds, generated_tokens = translate_one(model, row.source, row.src_lang, row.tgt_lang)
        new_rows.append({
            **row._asdict(),
            'model_name': 'public_v2_adapter',
            'prediction': prediction,
            'total_seconds': total_seconds,
            'generated_tokens': generated_tokens,
            'has_repetition': repeated_phrase(prediction),
            'hit_max_tokens': generated_tokens >= MAX_NEW_TOKENS,
        })
        if len(new_rows) % 25 == 0:
            saved = pd.concat([existing, pd.DataFrame(new_rows)], ignore_index=True)
            saved.drop_duplicates('eval_id', keep='last').sort_values('eval_id').to_csv(PREDICTION_FILE, index=False, encoding='utf-8-sig')
    predictions = pd.concat([existing, pd.DataFrame(new_rows)], ignore_index=True)
    predictions = predictions.drop_duplicates('eval_id', keep='last').sort_values('eval_id')
    predictions.to_csv(PREDICTION_FILE, index=False, encoding='utf-8-sig')
    del model
    gc.collect()
    torch.cuda.empty_cache()
else:
    predictions = existing.sort_values('eval_id')
assert len(predictions) == len(benchmark_df), f'Only {len(predictions)}/{len(benchmark_df)} predictions are complete.'
print('Saved:', PREDICTION_FILE)


Completed: 0 Pending: 1200


D:\dev\projects\fourlang_translation\.hf_compat\transformers\generation\configuration_utils.py:638: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(
public_v2_adapter:   2%|▏         | 25/1200 [00:12<10:13,  1.91it/s]D:\dev\projects\fourlang_translation\.hf_compat\transformers\generation\configuration_utils.py:638: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(
public_v2_adapter:   4%|▍         | 50/1200 [00:25<09:10,  2.09it/s]D:\dev\projects\fourlang_translation\.hf_compat\transformers\generation\configuration_utils.py:638: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. Y

Saved: D:\dev\projects\fourlang_translation\results\trusted_benchmark_public_v2\predictions_public_v2_adapter.csv


In [5]:
def truthy(series):
    if series.dtype == bool:
        return series
    return series.astype(str).str.lower().isin(['true', '1', 'yes'])

def compute_metrics(frame):
    records = []
    for (benchmark, src_lang, tgt_lang), group in frame.groupby(['benchmark', 'src_lang', 'tgt_lang']):
        preds = group['prediction'].astype(str).tolist()
        refs = group['reference'].astype(str).tolist()
        reference_lengths = group['reference'].astype(str).str.len().clip(lower=1)
        prediction_lengths = group['prediction'].astype(str).str.len()
        records.append({
            'model_name': str(group['model_name'].iloc[0]),
            'benchmark': benchmark,
            'direction': f'{src_lang}-{tgt_lang}',
            'samples': len(group),
            'bleu': sacrebleu.corpus_bleu(preds, [refs]).score,
            'chrf2': sacrebleu.corpus_chrf(preds, [refs], word_order=2).score,
            'repetition_percent': float(truthy(group['has_repetition']).mean() * 100),
            'hit_max_tokens_percent': float(truthy(group['hit_max_tokens']).mean() * 100),
            'mean_length_ratio': float((prediction_lengths / reference_lengths).mean()),
            'latency_mean_seconds': float(pd.to_numeric(group['total_seconds']).mean()),
            'latency_p95_seconds': float(pd.to_numeric(group['total_seconds']).quantile(.95)),
        })
    return pd.DataFrame(records)

new_metrics = compute_metrics(predictions)
new_metrics.to_csv(RESULT_DIR / 'metrics_public_v2_adapter.csv', index=False, encoding='utf-8-sig')
display(new_metrics.round(4))


,model_name,benchmark,direction,samples,bleu,chrf2,repetition_percent,hit_max_tokens_percent,mean_length_ratio,latency_mean_seconds,latency_p95_seconds
0,public_v2_adapter,flores_devtest,en-uz,200,1.8687,18.0263,1.0,1.5,0.7414,0.5237,0.9237
1,public_v2_adapter,flores_devtest,uz-en,200,4.5619,25.2030,0.5,0.0,0.9532,0.3867,0.6570
2,public_v2_adapter,ntrex,en-uz,200,1.2838,16.8644,0.0,2.0,0.7446,0.6098,1.4085
3,public_v2_adapter,ntrex,uz-en,200,3.2722,25.1977,0.0,0.0,0.9725,0.3712,0.7061
4,public_v2_adapter,tatoeba_latin,en-uz,200,0.8718,14.1429,0.0,0.0,0.8253,0.1583,0.2448
5,public_v2_adapter,tatoeba_latin,uz-en,200,5.8658,22.3478,0.0,0.0,0.9723,0.0993,0.1582


In [6]:
old_prediction_dir = PROJECT_ROOT / 'results' / 'trusted_benchmark_eval'
old_frames = []
for filename, model_name in [('predictions_base.csv', 'base'), ('predictions_adapter.csv', 'old_hplt_adapter')]:
    old_frame = pd.read_csv(old_prediction_dir / filename, keep_default_na=False)
    old_frame['model_name'] = model_name
    old_frame['has_repetition'] = old_frame['prediction'].map(repeated_phrase)
    old_frames.append(compute_metrics(old_frame))
old_metrics = pd.concat(old_frames, ignore_index=True)
all_metrics = pd.concat([old_metrics, new_metrics], ignore_index=True)
all_metrics.to_csv(RESULT_DIR / 'metrics_base_old_new.csv', index=False, encoding='utf-8-sig')

score_table = all_metrics.pivot_table(index=['benchmark', 'direction'], columns='model_name', values=['bleu', 'chrf2'])
display(score_table.round(4))

base = all_metrics[all_metrics.model_name == 'base'].set_index(['benchmark', 'direction'])
new = all_metrics[all_metrics.model_name == 'public_v2_adapter'].set_index(['benchmark', 'direction'])
comparison = new[['bleu', 'chrf2', 'repetition_percent', 'hit_max_tokens_percent', 'latency_p95_seconds']].subtract(
    base[['bleu', 'chrf2', 'repetition_percent', 'hit_max_tokens_percent', 'latency_p95_seconds']]
).add_prefix('delta_new_minus_base_').reset_index()
comparison.to_csv(RESULT_DIR / 'comparison_public_v2_minus_base.csv', index=False, encoding='utf-8-sig')
display(comparison.round(4))

summary = all_metrics.groupby(['model_name', 'direction'])[['bleu', 'chrf2', 'repetition_percent', 'hit_max_tokens_percent', 'latency_p95_seconds']].mean().reset_index()
summary.to_csv(RESULT_DIR / 'summary_mean_across_benchmarks.csv', index=False, encoding='utf-8-sig')
display(summary.round(4))


bleu                                       chrf2  \
model_name                  base old_hplt_adapter public_v2_adapter     base   
benchmark      direction                                                       
flores_devtest en-uz      1.2478           1.5830            1.8687  15.8707   
               uz-en      2.4271           4.8475            4.5619  22.2743   
ntrex          en-uz      0.9801           1.0561            1.2838  15.4115   
               uz-en      1.6972           3.2107            3.2722  21.9211   
tatoeba_latin  en-uz      1.1628           0.8775            0.8718  13.4519   
               uz-en      3.8229           4.5868            5.8658  15.9702   

                                                             
model_name               old_hplt_adapter public_v2_adapter  
benchmark      direction                                     
flores_devtest en-uz              17.7466           18.0263  
               uz-en              26.5388           25.2030  
ntrex          en-uz              16.2427           16.8644  
               uz-en              25.2369           25.1977  
tatoeba_latin  en-uz              13.6559           14.1429  
               uz-en              20.3427           22.3478

,benchmark,direction,delta_new_minus_base_bleu,delta_new_minus_base_chrf2,delta_new_minus_base_repetition_percent,delta_new_minus_base_hit_max_tokens_percent,delta_new_minus_base_latency_p95_seconds
0,flores_devtest,en-uz,0.6209,2.1556,-6.5,1.5,0.5189
1,flores_devtest,uz-en,2.1348,2.9288,0.0,-1.0,0.2008
2,ntrex,en-uz,0.3037,1.4528,-12.0,2.0,0.9521
3,ntrex,uz-en,1.5751,3.2766,0.0,-0.5,0.1227
4,tatoeba_latin,en-uz,-0.2909,0.6910,0.0,0.0,0.1473
5,tatoeba_latin,uz-en,2.0430,6.3776,0.0,0.0,0.0568


,model_name,direction,bleu,chrf2,repetition_percent,hit_max_tokens_percent,latency_p95_seconds
0,base,en-uz,1.1302,14.9114,6.5000,0.0000,0.3196
1,base,uz-en,2.6490,20.0552,0.1667,0.5000,0.3803
2,old_hplt_adapter,en-uz,1.1722,15.8817,0.5000,0.8333,0.7219
3,old_hplt_adapter,uz-en,4.2150,24.0394,0.0000,0.0000,0.4617
4,public_v2_adapter,en-uz,1.3415,16.3445,0.3333,1.1667,0.8590
5,public_v2_adapter,uz-en,4.5667,24.2495,0.1667,0.0000,0.5071


In [7]:
diagnostic = predictions.copy()
diagnostic['reference_chars'] = diagnostic['reference'].astype(str).str.len().clip(lower=1)
diagnostic['prediction_chars'] = diagnostic['prediction'].astype(str).str.len()
diagnostic['length_ratio'] = diagnostic['prediction_chars'] / diagnostic['reference_chars']
problem_rows = diagnostic[truthy(diagnostic['has_repetition']) | truthy(diagnostic['hit_max_tokens']) | (diagnostic['length_ratio'] < 0.35) | (diagnostic['length_ratio'] > 2.5)]
print('Diagnostic problem rows:', len(problem_rows), '/', len(diagnostic))
display(problem_rows[['benchmark', 'src_lang', 'tgt_lang', 'source', 'reference', 'prediction', 'length_ratio', 'has_repetition', 'hit_max_tokens']].head(30))

sample_rows = diagnostic.groupby(['benchmark', 'src_lang', 'tgt_lang'], group_keys=False).sample(n=3, random_state=42)
display(sample_rows[['benchmark', 'src_lang', 'tgt_lang', 'source', 'reference', 'prediction']].sort_values(['benchmark', 'src_lang']))

manifest = {
    'base_model': BASE_MODEL,
    'adapter_path': str(ADAPTER_PATH),
    'benchmark_file': str(BENCHMARK_FILE),
    'samples': int(len(benchmark_df)),
    'decode': {'num_beams': 1, 'repetition_penalty': 1.15, 'no_repeat_ngram_size': 3, 'max_new_tokens': MAX_NEW_TOKENS},
}
(RESULT_DIR / 'evaluation_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print('All trusted benchmark artifacts are ready in:', RESULT_DIR)


Diagnostic problem rows: 14 / 1200


,benchmark,src_lang,tgt_lang,source,reference,prediction,length_ratio,has_repetition,hit_max_tokens
26,flores_devtest,en,uz,Five minutes into the display a wind starts ro...,"Ko'rgazmadan besh daqiqa o'tib, shamol esa bos...","U ekranda 5 dakida r'zasi bo'zadi, o'zdan o'zg...",0.674419,False,True
33,flores_devtest,en,uz,"Her other race, the Giant Slalom, saw her fini...",U o'zining boshqa poygasi Gigant slalomda ayol...,"U o'zari, Giant Slalom, u o'zgari o'gari o`zar...",0.823529,False,True
75,flores_devtest,en,uz,Also between each dynasty was an unstable age ...,"Shuningdek, har bir sulola orasida beqaror par...",U q'yolda da o'ziylar bo'ilgan o'q'yogasi. Bu ...,1.134715,False,True
95,flores_devtest,en,uz,If you find yourself resetting the clock in yo...,Agar siz uyqudaligingizda soatni o'chirib qo'y...,"U u u u o'zini bo'zida ko'zidan bo'ilgan, o'yi...",0.719577,True,False
147,flores_devtest,en,uz,The official Falklands currency is the Falklan...,Folklendning rasmiy valyutasi — Folklend funti...,"Falkland valutasi - Falklandi pound (FKP), o'y...",0.817460,True,False
321,flores_devtest,uz,en,"Afsuski, yozuvning yangi usullari vujudga kelg...","Sadly, as newer methods of writing have emerge...","Afsuski, a former member of the Association fo...",1.076190,True,False
465,ntrex,en,uz,"""Part of what he's doing that makes it feel li...",“U bajarayotgan ishlarning voqelikni realiti-s...,"""O'yagan o'ynini reality showga o'zilgan o'yin...",0.653465,False,True
477,ntrex,en,uz,The last time the death penalty was carried ou...,Nyu Yorkda federal jinoyat ishi boʻyicha oʻlim...,U New York federal o'zida o'yashda o'ynagan o'...,0.954545,False,True
555,ntrex,en,uz,"To those who promoted the motion on Friday, al...",Juma kungi harakatda tashabbus ko‘rsatganlarga...,"U bu motifini o'yaganlarga, u o'ynamida o'yin ...",0.978571,False,True
562,ntrex,en,uz,Celebrity stylist Luke Armitage told FEMAIL: '...,Taniqli stilist Lyuke Armitaj FEMAIL nashriga ...,"FEMAIL-yilda o'zari stilist Luke Armitage: ""M'...",0.924658,False,True


,benchmark,src_lang,tgt_lang,source,reference,prediction
95,flores_devtest,en,uz,If you find yourself resetting the clock in yo...,Agar siz uyqudaligingizda soatni o'chirib qo'y...,"U u u u o'zini bo'zida ko'zidan bo'ilgan, o'yi..."
15,flores_devtest,en,uz,"With only eighteen medals available a day, a n...",Bir kunda bor-yo'g'i o'n sakkizta medal bo'lga...,"Bir günda o'zdan madalida bo'ilgan, bir sayta ..."
30,flores_devtest,en,uz,The composition of these crystals matches thos...,Bu kristallarning tuzilishi infraqizil spektro...,"Bu kristallar, infrared spektroskopiyasi (FTIR..."
376,flores_devtest,uz,en,"Bajarilgan ishlarning aksariyati nazariy edi, ...","The work done was mostly theoretical, but the ...","The resource of the resource was created, and ..."
233,flores_devtest,uz,en,U o'zining boshqa poygasi Gigant slalomda ayol...,"Her other race, the Giant Slalom, saw her fini...","The second time, the Austrian Claudia Loes, wh..."
321,flores_devtest,uz,en,"Afsuski, yozuvning yangi usullari vujudga kelg...","Sadly, as newer methods of writing have emerge...","Afsuski, a former member of the Association fo..."
560,ntrex,en,uz,The 56-year-old plays Jack Jarvis on the popul...,56 yashar aktyor BBCning mashhur serialida Jek...,"56-yil o'zari Jack Jarvis BBC va bo'zilgan, o'..."
414,ntrex,en,uz,"""He fully understood the role that he had toda...",“U bugun unga yuklangan vazifani toʻliq tushun...,"""Honning o'yogida o'ynagan rolini tamxilgan, u..."
509,ntrex,en,uz,And Nielsen usually has some trouble measuring...,"Shu bilan birga, Nielsen ish joylarida televiz...",Nielsen o'zilgan ofislarda o'ynagan o'yagan oʻ...
738,ntrex,uz,en,Janob Gendon Roa Roa mehmonxonasi qulagani haq...,"Mr. Gendon recounted how, in the hours after t...",Janob Gendon Roa Roa was a member of the Parap...


All trusted benchmark artifacts are ready in: D:\dev\projects\fourlang_translation\results\trusted_benchmark_public_v2
